# LeapSinger — Export the acoustic model to ONNX and use it

LeapSinger's acoustic model turns a **singing score** — phonemes, how long each one lasts, and an
**F0** (pitch) curve — into a **mel-spectrogram**. A separate **vocoder** then turns that mel into
audio.

This notebook exports the acoustic model to a **single-file ONNX** and runs it end to end: you give
the model phonemes, durations, and F0, and it returns a mel that a vocoder renders to audio. It runs
in a notebook or on any device that has ONNX Runtime.

To make the demo real, we drive it with one **ground-truth phrase** sung by *Namine Ritsu*: we take
the phoneme timing and the F0 from the recording, feed them to the model, and compare the result to
the original.

> Run this notebook from the repository root (the folder that contains `export/`, `preprocess/`,
> `leapsinger/`). It needs `onnxruntime`, `librosa`, `soundfile`, `matplotlib`, and `torch`
> (torch is only used to build the F0 extractor and to run the export).

In [ ]:
import os, sys, subprocess
import numpy as np

# Make the paths below work whether you launch from the repo root or from notebooks/.
ROOT = os.getcwd()
if os.path.basename(ROOT) == "notebooks":
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
sys.path.insert(0, ROOT)

SAMPLE_DIR = "notebooks/sample_data"      # the ground-truth phrase (wav + lab) + the model .pth
OUT_DIR    = "notebooks/exported_model"   # everything we export goes here
print("working directory:", ROOT)


## The acoustic model as a standalone ONNX

The exported ONNX is a single file with a small, clear signature:

- **Inputs**
  - `tokens` — phoneme IDs, shape `[1, n_phonemes]`
  - `durations` — number of frames each phoneme lasts, shape `[1, n_phonemes]`
  - `f0` — pitch in Hz, one value per frame, shape `[1, n_frames]`
- **Output**
  - `mel` — the mel-spectrogram, shape `[1, n_frames, 128]`

The harmonic + noise excitation and the single flow step are **baked into the graph**, so you
never build them yourself. You only provide phonemes, durations, and F0.

### Step 1 — Get the acoustic model checkpoint (`.pth`)

The model is shipped as a training checkpoint (see the project's GitHub Releases). A training
checkpoint also stores the optimizer state, which inference does not need. We keep only the weights
and the config, and save that as a small `.pth`:

```python
import torch
ck = torch.load("path/to/ckpt_030000.pt", map_location="cpu", weights_only=False)
torch.save({"model": ck["model"], "config": ck["config"],
            "n_speakers": ck["n_speakers"], "step": ck["step"]},
           "notebooks/sample_data/3speaker_gan2d.pth")
```

For this notebook, that `.pth` is already provided in `notebooks/sample_data/`.


In [ ]:
PTH = f"{SAMPLE_DIR}/3speaker_gan2d.pth"
assert os.path.exists(PTH), (
    f"{PTH} is missing. Create it from the training checkpoint as shown in Step 1.")
print("acoustic checkpoint:", PTH)


### Step 2 — Export to ONNX

We export a **single-speaker** graph for Namine Ritsu (speaker id `2`) at the model's native hop
size of 256.

- `--speaker bake --spk-id 2` freezes that one speaker into the graph, so the ONNX needs no
  speaker input.
- `--variant diffsinger` is the input contract `[tokens, durations, f0]` (this checkpoint is
  uv-free, so there is no separate voiced/unvoiced input).
- `--num-steps 1` runs the rectified flow in a **single step**. The flow starts from the excitation,
  which is already close to the target mel, so one straight step is enough — and works best here;
  adding steps only drifts away from the target.

In [ ]:
subprocess.run([
    sys.executable, "-m", "export.cli",
    "--ckpt", PTH,
    "--out", OUT_DIR,
    "--model-name", "leapsinger_ritsu_demo",
    "--variant", "diffsinger",
    "--hop", "256",
    "--num-steps", "1",        # rectified flow: one step (best here; more steps drift from target)
    "--speaker", "bake", "--spk-id", "2", "--spk-name", "ritsu",
], check=True)

ACOUSTIC_ONNX = f"{OUT_DIR}/leapsinger_ritsu_demo.ritsu.onnx"
print("exported:", ACOUSTIC_ONNX)

In [ ]:
# Confirm the input/output signature.
import onnx
g = onnx.load(ACOUSTIC_ONNX).graph
print("inputs :", [i.name for i in g.input])
print("outputs:", [o.name for o in g.output])


### Step 3 — Load the ground-truth phrase

`sample_data/` holds one phrase sung by Namine Ritsu:

- `ritsu_flashblack_0000.wav` — the raw recording (~19 s)
- `ritsu_flashblack_0000.lab` — the phoneme timing, one line per phoneme: `start end phoneme`
  (in seconds)

We will take the **phoneme timing** from the `.lab` and the **F0** from the `.wav`. To keep this
notebook small, we only use the first few seconds — cut at a silent `pau` so the clip ends cleanly.


In [ ]:
import librosa
from leapsinger.config import MelSpec
from preprocess.lab import load_lab

mel_cfg = MelSpec(hop=256)
SR, HOP, FPS = mel_cfg.sr, mel_cfg.hop, mel_cfg.frame_rate   # 44100, 256, 172.27 frames/sec

# The phrase is ~19 s. To keep the notebook small once the audio is embedded, we use only the
# first part. We cut at a silence (a `pau`, which also covers breaths) near TARGET_SECONDS, so the
# clip ends cleanly instead of in the middle of a word. Raise TARGET_SECONDS to hear more.
TARGET_SECONDS = 6.0
lab_rows = load_lab(f"{SAMPLE_DIR}/ritsu_flashblack_0000.lab", lab_unit="sec")
pau_ends = [e for s, e, ph in lab_rows if ph == "pau" and e > 1.0]
cut_sec = min(pau_ends, key=lambda t: abs(t - TARGET_SECONDS)) if pau_ends else None

wav_gt, _ = librosa.load(f"{SAMPLE_DIR}/ritsu_flashblack_0000.wav", sr=SR, mono=True,
                         duration=cut_sec)
n_frames = len(wav_gt) // HOP
rows = [(s, min(e, n_frames / FPS), ph) for s, e, ph in lab_rows if s < n_frames / FPS]
print(f"cut at a pause: {len(wav_gt) / SR:.1f} s, {n_frames} frames, {len(rows)} phonemes")


### Step 4 — Compute F0 and the mel from the audio

The model needs an F0 curve. We extract it with **RMVPE**, the F0 method this repo uses. We also
compute the mel of the recording, only so we can plot it next to the model's output at the end.

`interpolate=True` fills the unvoiced gaps, giving a **continuous** F0 — this model was trained on a
gap-less pitch curve.

> The first run downloads the RMVPE weights (~180 MB). Set `DEVICE` to `"cuda"` or `"mps"` if you
> have a GPU; it is much faster than `"cpu"`.


In [ ]:
from preprocess.f0_rmvpe import extract_f0_rmvpe
from leapsinger.mel import wav_to_mel_nhv

DEVICE = "cpu"

f0, voiced = extract_f0_rmvpe(np.clip(wav_gt, -1.0, 1.0), SR, HOP,
                              fmin=150.0, fmax=1000.0, device=DEVICE, interpolate=True)
f0, voiced = f0[:n_frames], voiced[:n_frames]
logf0 = np.log2(np.maximum(f0, 1.0)).astype(np.float32)     # the vocoder wants log2-F0

mel_gt = wav_to_mel_nhv(wav_gt, SR, mel_cfg.n_fft, HOP, mel_cfg.win,
                        mel_cfg.n_mels, mel_cfg.fmin, mel_cfg.fmax)[:, :n_frames]
print("F0 frames:", f0.shape[0], "| recording mel:", mel_gt.shape)


### Step 5 — Turn the label into phoneme IDs and durations

Each phoneme becomes an integer id (the model's vocabulary), and each phoneme's length becomes a
number of frames. The frame counts must add up to the number of F0 frames.


In [ ]:
from preprocess.vocab import PHONEME2ID

# `rows` was loaded from the .lab and trimmed to the cut point back in Step 3.
phonemes = [ph for _, _, ph in rows]
tokens = np.array([[PHONEME2ID[ph] for ph in phonemes]], dtype=np.int64)

# phoneme end-times -> frame boundaries -> per-phoneme frame counts (sum stays exact)
ends = np.cumsum([e - s for s, e, _ in rows])
fb = np.clip(np.round(np.concatenate([[0.0], ends]) * FPS).astype(int), 0, n_frames)
fb[-1] = n_frames
durations = np.maximum(np.diff(fb), 1)
durations[np.argmax(durations)] += n_frames - durations.sum()
durations = durations[None].astype(np.int64)

print(f"{len(phonemes)} phonemes; durations sum to {int(durations.sum())} (== {n_frames})")
print("first phonemes:", " ".join(phonemes[:16]))


### Step 6 — Run the acoustic model

Give the ONNX the tokens, durations, and F0. It returns the mel.


In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(os.path.abspath(ACOUSTIC_ONNX),
                            providers=["CPUExecutionProvider"])
mel_pred = sess.run(["mel"], {
    "tokens":    tokens,
    "durations": durations,
    "f0":        f0[None].astype(np.float32),
})[0][0].T   # ONNX emits the [T, mel] layout; transpose to [mel, T] for the plot + vocoder helper
print("predicted mel:", mel_pred.shape)

### Step 7 — Vocoder: mel → audio

The mel is turned into a waveform by the bundled NHVSing vocoder (`checkpoints/nhv_v3_1.onnx`). It takes
the mel plus the same F0 and voicing.


In [ ]:
from infer import load_vocoder, mel_to_wav

vocoder = load_vocoder("checkpoints/nhv_v3_1.onnx")
wav_pred = mel_to_wav(vocoder, mel_pred, logf0, voiced)
print("generated audio:", f"{len(wav_pred) / SR:.1f} s")


### Step 8 — Compare

Listen to both, and look at the two mels. The model was given only the **phoneme timing** and the
**F0** of the recording, so the result should closely follow the original singing.


In [ ]:
from IPython.display import Audio, display

print("Original recording")
display(Audio(wav_gt, rate=SR))
print("LeapSinger (from phoneme timing + F0)")
display(Audio(wav_pred, rate=SR))


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for ax, m, title in [(axes[0], mel_gt, "Recording — mel"),
                     (axes[1], mel_pred, "LeapSinger — mel")]:
    ax.imshow(m, origin="lower", aspect="auto", cmap="magma", vmin=-11, vmax=1)
    ax.set_ylabel("mel bin"); ax.set_title(title)
axes[1].set_xlabel("frame")
fig.tight_layout(); plt.show()


That is the whole standalone pipeline:

> **phoneme IDs + durations + F0**  →  *(acoustic ONNX)*  →  **mel**  →  *(vocoder ONNX)*  →  **audio**

Nothing above is specific to any particular host — any program with ONNX Runtime can do the same.

## Summary

- The acoustic model exports to a **single ONNX**: **phonemes + durations + F0 → mel**. Run it
  anywhere ONNX Runtime runs, then pair it with an NHVSing vocoder for audio.
- **Baking** a speaker (`--speaker bake`) gives the simplest graph (`tokens, durations, f0`).
  **Embedding** speakers (`--speaker embed`) adds a `spk_embed` vector, so one graph can sing as any
  speaker.